# Multi-agent systems: from prompts to executable actions
**Aadi / Rituparn Shukla · 60–120 minute webinar**

We will build a study schedule using three roles: Researcher, Planner and Reviewer.
The Python coordinator runs them in order and decides whether to accept or revise.

**Default mode is a labelled simulation. It does not call an LLM.**
The same runner supports real OpenAI-compatible model inference once configured.


## 1. Find the code
Run from the `webinar` folder, its `notebooks` folder, or the repository root.
Python 3.10+ is enough for the demo. No pip installation is needed.


In [ ]:
from pathlib import Path
import sys
import json
import inspect

candidates = [p / suffix for p in [Path.cwd(), *Path.cwd().parents]
              for suffix in ["", "webinar", "multi-agent-webinar"]]
ROOT = next((p for p in candidates if (p / "mas_demo" / "runner.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Extract the full webinar folder, then run this notebook from inside it.")
sys.path.insert(0, str(ROOT))

from mas_demo import prompts
from mas_demo.backends import ScriptedBackend, CompatibleBackend
from mas_demo.pipeline import Task, run_demo
from mas_demo.tools import DOCUMENTS, SEARCH_SCHEMA, validate_plan
from mas_demo.runner import run_agent
from mas_demo.report import save_report
print("Code folder:", ROOT.name)
print("Default backend:", ScriptedBackend.mode)


## 2. Start with a problem whose answer can be checked
Four lesson fixtures require 75 minutes. The student has 3 days × 45 minutes.
The plan must cover every topic, respect each day and cite the right source IDs.

**Audience prediction:** must the schedule fill all 135 available minutes?


In [ ]:
for document in DOCUMENTS:
    print(f"{document['id']}: {document['topic']} ({document['minutes']} min)")
print("Required:", sum(d['minutes'] for d in DOCUMENTS))
print("Capacity:", 3 * 45)


## 3. What belongs to an agent?
A model, role instructions, message context, permitted tools and a bounded runtime loop.
An agent name is a label. Separate roles do not require different underlying models.

| Role | Allowed function | Final output |
| --- | --- | --- |
| Researcher | search_documents | Retrieved source IDs |
| Planner | get_budget | Schedule |
| Reviewer | validate_plan | Verdict and issues |

This example is sequential and code-controlled. Context moves explicitly between roles.


## 4. Show the actual prompts
These are the instructions sent in system messages in live mode.
The simulated backend ignores them; prompt experiments require live inference.


In [ ]:
for name in ["RESEARCHER", "PLANNER", "REVIEWER"]:
    print("\n" + name + "\n")
    print(getattr(prompts, name))


## 5. Show a function schema
The model requests a function call. Python checks and executes it.
The tool response becomes an observation for the next model request.


In [ ]:
print(json.dumps(SEARCH_SCHEMA, indent=2))
print(inspect.getsource(run_agent))


## 6. Run the normal case
Follow the event stream: role start → model request → tool call → result → final JSON.
Visible events are not private model reasoning.


In [ ]:
def show(event):
    if event['kind'] == 'start':
        print(f"{event['seq']:02d} {event['agent']}: allowed tools {event['data']['allowed_tools']}")
    elif event['kind'] in ['tool_call', 'tool_result', 'review', 'revision_requested', 'injected_fault']:
        print(f"{event['seq']:02d} {event['agent']} / {event['kind']}: {json.dumps(event['data'])}")

normal = run_demo(Task(), ScriptedBackend(), observer=show)
print("MODE:", normal['mode'])
print("STATUS:", normal['status'])
print(json.dumps(normal['plan'], indent=2))
save_report(normal, ROOT / 'runs' / 'notebook-normal')


## 7. Demonstrate a failure and one revision
We deliberately corrupt the first session. This is fault injection by the presenter,
not a model-generated error. Identify the two failed constraints before the revision.


In [ ]:
repaired = run_demo(Task(), ScriptedBackend(), inject_bad_draft=True, observer=show)
reviews = [e['data'] for e in repaired['trace'] if e['kind'] == 'review']
print("First review:", reviews[0])
print("Last review:", reviews[-1])
print("Status:", repaired['status'], "Revisions:", repaired['revisions'])
save_report(repaired, ROOT / 'runs' / 'notebook-revision')


## 8. Inspect the feedback that crosses the role boundary
Each planner run gets a fresh message history. The coordinator explicitly transfers
the prior plan and concrete issues into the next run.


In [ ]:
planner_inputs = [e['data']['input'] for e in repaired['trace']
                  if e['agent'] == 'Planner' and e['kind'] == 'start']
print("Feedback in second planner run:")
print(json.dumps(planner_inputs[1]['feedback'], indent=2))


## 9. An impossible task should fail usefully
One day × 30 minutes cannot hold 75 minutes of required lessons.
The coordinator detects this before running the planner.


In [ ]:
impossible = run_demo(Task(days=1, daily_minutes=30), ScriptedBackend())
print(impossible['status'])
print(impossible['message'])
assert impossible['plan'] is None

unrepaired = run_demo(Task(), ScriptedBackend(), inject_bad_draft=True, max_revisions=0)
print("No revisions allowed:", unrepaired['status'])


## 10. Optional: switch to a real local LLM
Install Ollama, run `ollama pull qwen3:8b`, and start the Ollama app first.
This is an **optional live call**, not a fixture. Run the next cell intentionally.
If you use another provider, set its endpoint, model and API key in your shell before opening Jupyter.


In [ ]:
# Run this cell only after you have started Ollama and installed qwen3:8b.
import os
os.environ["LLM_BASE_URL"] = "http://localhost:11434/v1"
os.environ["LLM_MODEL"] = "qwen3:8b"
os.environ["LLM_TIMEOUT_SECONDS"] = "300"
live_backend = CompatibleBackend.from_env()
live = run_demo(Task(), live_backend, observer=show)
print("MODE:", live["mode"], "STATUS:", live["status"])
print("Requests:", live_backend.requests, "Usage:", live_backend.usage)
save_report(live, ROOT / "runs" / "notebook-live")


## 11. Student experiment
Predict the results of two days × 30 minutes and three days × 30 minutes.
Which one needs a change of scope? Which one can split topics across days?


In [ ]:
for days in [2, 3]:
    result = run_demo(Task(days=days, daily_minutes=30), ScriptedBackend())
    print(f"{days} days × 30 minutes: {result['status']}")


## 12. What did we verify?
Offline tests check actual tools, evidence transfer, budget, coverage and stopping.
A local HTTP fixture checks the compatible protocol. Neither checks real LLM quality.
For that, run repeated live cases and compare with a single-agent baseline.

## Continue learning
- [Hugging Face agent introduction](https://huggingface.co/learn/agents-course/unit1/what-are-agents)
- [Anthropic workflow patterns](https://www.anthropic.com/engineering/building-effective-agents)
- [OpenAI orchestration](https://developers.openai.com/api/docs/guides/agents/orchestration)
- [Teaching guide](../docs/teaching-guide.md)
- [Exercises](../docs/exercises.md)
